<a href="https://colab.research.google.com/github/JacekDrzycimski/6tunnel/blob/master/Quaternions_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyquaternion

In [2]:
from pyquaternion import Quaternion
import numpy as np

# 1. Create quaternions
# Creating a quaternion representing a 90-degree rotation around the Z-axis
q1 = Quaternion(axis=[0, 0, 1], angle=np.pi/2)
print(f"q1 (90 deg around Z): {q1}")

# 2. Quaternion multiplication (combining rotations)
q2 = Quaternion(axis=[1, 0, 0], angle=np.pi/4) # 45 deg around X-axis
q_combined = q1 * q2
print(f"Combined rotation q_combined: {q_combined}")

# 3. Rotating a vector
v = np.array([1, 0, 0])
v_rotated = q1.rotate(v)
print(f"Vector {v} rotated by q1: {v_rotated}")

# 4. Converting to a rotation matrix
rot_matrix = q1.rotation_matrix
print("Rotation matrix:\n", rot_matrix)

q1 (90 deg around Z): +0.707 +0.000i +0.000j +0.707k
Combined rotation q_combined: +0.653 +0.271i +0.271j +0.653k
Vector [1 0 0] rotated by q1: [2.22044605e-16 1.00000000e+00 0.00000000e+00]
Rotation matrix:
 [[ 1.99673462e-16 -1.00000000e+00  0.00000000e+00]
 [ 1.00000000e+00  1.79380389e-16  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [3]:
from scipy.spatial.transform import Rotation as R
import numpy as np

# 1. Convert Euler angles to a Quaternion
# Let's define Euler angles: Yaw = 90 deg, Pitch = 45 deg, Roll = 30 deg
# We specify the sequence 'zyx' (commonly used in aerospace/navigation)
euler_degrees = [90, 45, 30]

# Create a rotation object from the Euler angles
r = R.from_euler('zyx', euler_degrees, degrees=True)

# Get the quaternion representation [x, y, z, w]
# Note: SciPy outputs quaternions in [x, y, z, w] format
q_scipy = r.as_quat()
print(f"Euler angles (Yaw/Pitch/Roll): {euler_degrees}")
print(f"Corresponding Quaternion [x, y, z, w]: {q_scipy}\n")

# 2. Convert Quaternion back to Euler angles
# Recreate a rotation object from our quaternion
r_back = R.from_quat(q_scipy)

# Retrieve Euler angles back in the 'zyx' sequence
euler_back = r_back.as_euler('zyx', degrees=True)
print(f"Converted back to Euler angles (Yaw/Pitch/Roll): {euler_back}")

Euler angles (Yaw/Pitch/Roll): [90, 45, 30]
Corresponding Quaternion [x, y, z, w]: [0.43045933 0.09229596 0.70105738 0.56098553]

Converted back to Euler angles (Yaw/Pitch/Roll): [90. 45. 30.]


In [2]:
!pip install -q pyquaternion
import time
from scipy.spatial.transform import Rotation as R
from pyquaternion import Quaternion
import numpy as np

# Number of iterations for our benchmark
N = 50000

print(f"Benchmarking {N} iterations...\n")

# --- 1. PYQUATERNION BENCHMARK ---
start_time = time.time()
for _ in range(N):
    # Create quaternion and rotate vector
    q = Quaternion(axis=[0, 0, 1], angle=np.pi/2)
    v_rot = q.rotate([1.0, 0.0, 0.0])
pyquat_time = time.time() - start_time
print(f"pyquaternion time: {pyquat_time:.4f} seconds")

# --- 2. SCIPY BENCHMARK ---
start_time = time.time()
for _ in range(N):
    # Create rotation object and rotate vector
    r = R.from_euler('z', 90, degrees=True)
    v_rot = r.apply([1.0, 0.0, 0.0])
scipy_time = time.time() - start_time
print(f"SciPy (individual loops) time: {scipy_time:.4f} seconds")

# --- 3. SCIPY BATCHED BENCHMARK (The vectorized way) ---
# SciPy shines when batching computations instead of looping in Python
start_time = time.time()
angles = np.ones(N) * 90
vectors = np.tile([1.0, 0.0, 0.0], (N, 1))
r_batch = R.from_euler('z', angles, degrees=True)
v_rot_batch = r_batch.apply(vectors)
scipy_batch_time = time.time() - start_time
print(f"SciPy (batched/vectorized) time: {scipy_batch_time:.4f} seconds")

# Print speedup
print(f"\nBatched SciPy is {pyquat_time / scipy_batch_time:.1f}x faster than pyquaternion!")

Benchmarking 50000 iterations...

pyquaternion time: 0.8970 seconds
SciPy (individual loops) time: 0.2791 seconds
SciPy (batched/vectorized) time: 0.0040 seconds

Batched SciPy is 221.9x faster than pyquaternion!
